# Notebook 06 — GraphSAGE training & inductive evaluation

**Tasks 2.1–2.3 (TODO):** heterogeneous GraphSAGE encoder, training loop with `LinkNeighborLoader`, checkpointing, then **inductive** evaluation on hidden users (`hidden_interactions_test.parquet`).

**Prerequisites:** run notebooks `01`–`05` so `data/processed/` contains `hetero_data.pt`, `hidden_interactions_test.parquet`, and feature tensors.

**Encoder:** two `HeteroConv` stacks of `SAGEConv` (heterogeneous GraphSAGE), matching each edge type including `rev_reviews`.

**Note:** The saved graph only has `user → item` edges. For bipartite message passing, we add **`item → user` reverse edges** (flip of `edge_index`) on each split so users receive item signals — standard in heterogeneous GNN link prediction.

In [7]:
import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.transforms import RandomLinkSplit

PROCESSED_DIR = Path("../data/processed/")
CKPT_PATH = PROCESSED_DIR / "graphsage_link_pred.pt"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

device: cpu
torch: 2.11.0+cu130


In [8]:
class DotProductDecoder(nn.Module):
    """From notebook 05 — logits [batch] = sum_d u_d * v_d."""

    def forward(self, user_emb: torch.Tensor, item_emb: torch.Tensor) -> torch.Tensor:
        return (user_emb * item_emb).sum(dim=-1)


class HeteroSAGEEncoder(nn.Module):
    """2-layer hetero GraphSAGE: user↔item (both directions) + item↔item."""

    def __init__(
        self,
        metadata: tuple,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        edge_types = metadata[1]
        convs1 = {}
        convs2 = {}
        for et in edge_types:
            src, _, dst = et
            if src == "item" and dst == "user":
                convs1[et] = SAGEConv((in_channels, in_channels), hidden_channels, aggr="mean")
                convs2[et] = SAGEConv(
                    (hidden_channels, hidden_channels), out_channels, aggr="mean"
                )
            elif src == "user" and dst == "item":
                convs1[et] = SAGEConv((in_channels, in_channels), hidden_channels, aggr="mean")
                convs2[et] = SAGEConv(
                    (hidden_channels, hidden_channels), out_channels, aggr="mean"
                )
            elif src == "item" and dst == "item":
                convs1[et] = SAGEConv(in_channels, hidden_channels, aggr="mean")
                convs2[et] = SAGEConv(hidden_channels, out_channels, aggr="mean")
            else:
                raise ValueError(f"Unexpected edge type: {et}")
        self.convs1 = HeteroConv(convs1, aggr="sum")
        self.convs2 = HeteroConv(convs2, aggr="sum")

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.convs1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(x) for k, x in x_dict.items()}
        x_dict = {k: self.dropout(x) for k, x in x_dict.items()}
        x_dict = self.convs2(x_dict, edge_index_dict)
        return x_dict

In [9]:
def add_reverse_reviews(data: HeteroData) -> HeteroData:
    """Add (item, rev_reviews, user) = flip of (user, reviews, item) for message passing."""
    out = data.clone()
    rev = out["user", "reviews", "item"].edge_index.flip(0)
    out["item", "rev_reviews", "user"].edge_index = rev
    return out


def decode_link_logits(batch, z_dict, decoder):
    """batch: hetero mini-batch; z_dict: encoder output (local node indices)."""
    edge_label_index = batch["user", "reviews", "item"].edge_label_index
    src_user = edge_label_index[0]
    src_item = edge_label_index[1]
    z_u = z_dict["user"][src_user]
    z_i = z_dict["item"][src_item]
    return decoder(z_u, z_i)


@torch.no_grad()
def eval_loader_metrics(encoder, decoder, loader, device):
    encoder.eval()
    decoder.eval()
    ys, scores = [], []
    for batch in loader:
        batch = batch.to(device)
        z_dict = encoder(batch.x_dict, batch.edge_index_dict)
        logits = decode_link_logits(batch, z_dict, decoder)
        y = batch["user", "reviews", "item"].edge_label.float()
        ys.append(y.cpu())
        scores.append(logits.cpu())
    y = torch.cat(ys).numpy()
    s = torch.cat(scores).numpy()
    out = {"n": int(y.size), "positives": int(y.sum()), "negatives": int((1 - y).sum())}
    if out["positives"] > 0 and out["negatives"] > 0:
        out["auroc"] = float(roc_auc_score(y, s))
        out["ap"] = float(average_precision_score(y, s))
        out["acc_0.5"] = float(((s >= 0) == y).mean())  # threshold 0 on logits
    else:
        out["auroc"] = float("nan")
        out["ap"] = float("nan")
        out["acc_0.5"] = float("nan")
    return out

In [10]:
# --- Load graph (seen users only) — same as notebook 04 ---
data = torch.load(PROCESSED_DIR / "hetero_data.pt", weights_only=False)
print(data)

assert (
    "item",
    "rev_reviews",
    "user",
) not in data.edge_types, "Re-run without duplicate rev edges in hetero_data.pt"

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.2,
    neg_sampling_ratio=0.0,
    add_negative_train_samples=False,
    edge_types=("user", "reviews", "item"),
    rev_edge_types=None,
)
train_data, val_data, test_data = transform(data)
train_data = add_reverse_reviews(train_data)
val_data = add_reverse_reviews(val_data)
test_data = add_reverse_reviews(test_data)

metadata = train_data.metadata()
print("metadata (with rev):", metadata)

IN_CHANNELS = data["user"].x.size(1)
HIDDEN = 256
OUT_CH = 256
DROPOUT = 0.2
BATCH_SIZE = 256
NUM_NEIGHBORS = [15, 10]
NEG_SAMPLING_RATIO = 1.0

train_loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=(
        ("user", "reviews", "item"),
        train_data["user", "reviews", "item"].edge_label_index,
    ),
    edge_label=train_data["user", "reviews", "item"].edge_label,
    neg_sampling_ratio=NEG_SAMPLING_RATIO,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

# val: 1 neg / pos so AUROC/AP are defined (supervision edges are all positives from RandomLinkSplit)
val_loader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=(
        ("user", "reviews", "item"),
        val_data["user", "reviews", "item"].edge_label_index,
    ),
    edge_label=val_data["user", "reviews", "item"].edge_label,
    neg_sampling_ratio=1.0,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader = LinkNeighborLoader(
    data=test_data,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=(
        ("user", "reviews", "item"),
        test_data["user", "reviews", "item"].edge_label_index,
    ),
    edge_label=test_data["user", "reviews", "item"].edge_label,
    neg_sampling_ratio=1.0,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

HeteroData(
  user={ x=[1642, 384] },
  item={ x=[21639, 384] },
  (user, reviews, item)={ edge_index=[2, 10751] },
  (item, also_bought, item)={ edge_index=[2, 7253] }
)
metadata (with rev): (['user', 'item'], [('user', 'reviews', 'item'), ('item', 'also_bought', 'item'), ('item', 'rev_reviews', 'user')])


In [11]:
# --- Sanity: one batch, local edge_label_index inside subgraph ---
encoder = HeteroSAGEEncoder(metadata, IN_CHANNELS, HIDDEN, OUT_CH, DROPOUT).to(device)
decoder = DotProductDecoder().to(device)

batch = next(iter(train_loader)).to(device)
z = encoder(batch.x_dict, batch.edge_index_dict)
el = batch["user", "reviews", "item"].edge_label_index
assert el[0].max() < z["user"].size(0) and el[1].max() < z["item"].size(0)
logits = decode_link_logits(batch, z, decoder)
assert torch.isfinite(logits).all()
assert z["user"].size(1) == z["item"].size(1) == OUT_CH
print("Sanity OK — shapes:", {k: tuple(t.shape) for k, t in z.items()}, "logits", logits.shape)

Sanity OK — shapes: {'item': (1396, 256), 'user': (1511, 256)} logits torch.Size([512])


In [12]:
def train_epochs(encoder, decoder, train_loader, val_loader, epochs=5, lr=1e-3, wd=0.0):
    encoder.train()
    decoder.train()
    params = list(encoder.parameters()) + list(decoder.parameters())
    opt = torch.optim.Adam(params, lr=lr, weight_decay=wd)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(1, epochs + 1):
        encoder.train()
        decoder.train()
        total_loss = 0.0
        n_batches = 0
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            z_dict = encoder(batch.x_dict, batch.edge_index_dict)
            logits = decode_link_logits(batch, z_dict, decoder)
            y = batch["user", "reviews", "item"].edge_label.float()
            loss = loss_fn(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            opt.step()
            total_loss += loss.item()
            n_batches += 1
        train_loss = total_loss / max(n_batches, 1)

        val_m = eval_loader_metrics(encoder, decoder, val_loader, device)
        print(
            f"Epoch {epoch:02d}  train_loss={train_loss:.4f}  "
            f"val AUROC={val_m['auroc']:.4f}  val AP={val_m['ap']:.4f}  "
            f"val acc@0.5={val_m['acc_0.5']:.4f}"
        )

    return encoder, decoder


EPOCHS = 5  # increase for reports / GPU runs
encoder, decoder = train_epochs(
    encoder, decoder, train_loader, val_loader, epochs=EPOCHS, lr=1e-3
)

Epoch 01  train_loss=0.4485  val AUROC=0.9814  val AP=0.9703  val acc@0.5=0.9786
Epoch 02  train_loss=0.1330  val AUROC=0.9840  val AP=0.9790  val acc@0.5=0.9814
Epoch 03  train_loss=0.1188  val AUROC=0.9849  val AP=0.9799  val acc@0.5=0.9823
Epoch 04  train_loss=0.0864  val AUROC=0.9840  val AP=0.9721  val acc@0.5=0.9800
Epoch 05  train_loss=0.0810  val AUROC=0.9828  val AP=0.9706  val acc@0.5=0.9753


In [13]:
# --- Seen-user test split (edge-level; same users as train) ---
test_seen = eval_loader_metrics(encoder, decoder, test_loader, device)
print("Test (seen users, edge split):", test_seen)

torch.save(
    {
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "metadata": metadata,
        "hparams": {
            "in_channels": IN_CHANNELS,
            "hidden_channels": HIDDEN,
            "out_channels": OUT_CH,
            "dropout": DROPOUT,
            "seed": SEED,
            "num_neighbors": NUM_NEIGHBORS,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
        },
        "test_seen_users_edge_split": test_seen,
    },
    CKPT_PATH,
)
print("Saved checkpoint to", CKPT_PATH)

Test (seen users, edge split): {'n': 2150, 'positives': 1075, 'negatives': 1075, 'auroc': 0.9816566792861006, 'ap': 0.9719533486429255, 'acc_0.5': 0.9804651162790697}
Saved checkpoint to ../data/processed/graphsage_link_pred.pt


## Inductive holdout users (~10% from notebook 01)

Users here **never appear** in `hetero_data.pt`. We build a separate `HeteroData` with **local** user indices `0..H-1` and **global** item indices matching `item_mapping` / `x_item`. Item–item edges are the **same** as Cloud catalog.

In [14]:
hidden_path = PROCESSED_DIR / "hidden_interactions_test.parquet"
if not hidden_path.is_file():
    raise FileNotFoundError(
        f"Missing {hidden_path} — run notebook 01_data_engineering to export hidden interactions."
    )

with open(PROCESSED_DIR / "user_mapping.json") as f:
    seen_user_ids = set(json.load(f).keys())

hdf = pd.read_parquet(hidden_path)
with open(PROCESSED_DIR / "item_mapping.json") as f:
    item_mapping = json.load(f)

x_item_full = data["item"].x.clone()
ii_global = data["item", "also_bought", "item"].edge_index.clone()

hidden_users = hdf["reviewerID"].unique().tolist()
for uid in hidden_users:
    assert uid not in seen_user_ids, f"Hidden user {uid} overlaps seen training users"

hidden_user_map = {u: i for i, u in enumerate(hidden_users)}
H = len(hidden_user_map)

rows = []
dropped_oov = 0
for _, row in hdf.iterrows():
    asin = row["asin"]
    if asin not in item_mapping:
        dropped_oov += 1
        continue
    rows.append(
        {"u": hidden_user_map[row["reviewerID"]], "i": int(item_mapping[asin])}
    )

print(f"Hidden users H={H}, interaction rows kept={len(rows)}, dropped OOV asin={dropped_oov}")
assert len(rows) > 0, "No inductive edges after filtering"

src = torch.tensor([r["u"] for r in rows], dtype=torch.long)
dst = torch.tensor([r["i"] for r in rows], dtype=torch.long)
ui_ind = torch.stack([src, dst], dim=0)

data_ind = HeteroData()
data_ind["user"].x = torch.zeros(H, IN_CHANNELS, dtype=torch.float)
data_ind["item"].x = x_item_full
data_ind["user", "reviews", "item"].edge_index = ui_ind
data_ind["item", "also_bought", "item"].edge_index = ii_global

print(data_ind)

Hidden users H=183, interaction rows kept=1200, dropped OOV asin=0
HeteroData(
  user={ x=[183, 384] },
  item={ x=[21639, 384] },
  (user, reviews, item)={ edge_index=[2, 1200] },
  (item, also_bought, item)={ edge_index=[2, 7253] }
)


In [15]:
# Disjoint edge split on inductive graph only (same hyperparams style as 04)
ind_transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.2,
    neg_sampling_ratio=0.0,
    add_negative_train_samples=False,
    edge_types=("user", "reviews", "item"),
    rev_edge_types=None,
)
ind_train, ind_val, ind_test = ind_transform(data_ind)
ind_train = add_reverse_reviews(ind_train)
ind_val = add_reverse_reviews(ind_val)
ind_test = add_reverse_reviews(ind_test)

assert set(ind_train.edge_types) == set(metadata[1]) and set(ind_train.node_types) == set(
    metadata[0]
), "Inductive schema must match training (incl. rev_reviews)"

ind_test_loader = LinkNeighborLoader(
    data=ind_test,
    num_neighbors=NUM_NEIGHBORS,
    edge_label_index=(
        ("user", "reviews", "item"),
        ind_test["user", "reviews", "item"].edge_label_index,
    ),
    edge_label=ind_test["user", "reviews", "item"].edge_label,
    neg_sampling_ratio=1.0,
    batch_size=min(BATCH_SIZE, 128),
    shuffle=False,
    num_workers=0,
)

inductive_metrics = eval_loader_metrics(encoder, decoder, ind_test_loader, device)
print("Inductive test (hidden users):", inductive_metrics)

# Random-score baseline (same labels; expect AUROC near 0.5)
ys_base = []
encoder.eval()
decoder.eval()
with torch.no_grad():
    for batch in ind_test_loader:
        batch = batch.to(device)
        y = batch["user", "reviews", "item"].edge_label.float()
        ys_base.append(y.cpu())
y_base = torch.cat(ys_base).numpy()
rng = np.random.default_rng(SEED)
rnd = rng.standard_normal(y_base.shape)
if y_base.sum() > 0 and (1 - y_base).sum() > 0:
    baseline_auroc = float(roc_auc_score(y_base, rnd))
else:
    baseline_auroc = float("nan")
print("Random baseline AUROC:", baseline_auroc)

pack = torch.load(CKPT_PATH, weights_only=False)
pack["inductive_hidden_user_test"] = inductive_metrics
pack["inductive_baseline_random_auroc"] = baseline_auroc
torch.save(pack, CKPT_PATH)
print("Checkpoint updated with inductive metrics:", CKPT_PATH)


Inductive test (hidden users): {'n': 240, 'positives': 120, 'negatives': 120, 'auroc': 0.9361805555555556, 'ap': 0.9202146585685207, 'acc_0.5': 0.7916666666666666}
Random baseline AUROC: 0.5127777777777778
Checkpoint updated with inductive metrics: ../data/processed/graphsage_link_pred.pt
